# 02 — Offline Train and Evaluation Preparation

Prepare this policy’s frozen S20 and development images once from original sources, with shared five-class labels and JPEG95. Default mode reads the verified historical exports.

**Inputs:** Experiment YAML, shared frozen table, original sources for new exports; historical exports for read-only replay.

**Outputs:** runs/v1/data/{images,labels,data.yaml} only when ALLOW_PREPARATION=True.

**Prerequisites:** Saved config/table and explicit source paths. Never runs notebook 00 or calibrates a policy.

**Outline**

- [1. Configuration](#section-1)
- [2. Offline Inputs](#section-2)
- [3. Preparation Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

Choose historical S10 inspection or explicitly enable the new S20 offline export.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from vehicle_detection.common import read_json, require_paths

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
from vehicle_detection.experiment_data import load_experiment

EXPERIMENT = "v1"
require_paths(config=ROOT / f"configs/{EXPERIMENT}.yaml", shared=ROOT / "configs/data.yaml")
config = load_experiment(ROOT, EXPERIMENT)
from vehicle_detection.experiment_data import populations, prepare_data, shared_splits
from vehicle_detection.offline import verify_alignment

ALLOW_PREPARATION = False
HISTORICAL = {
    "train": ROOT / "runs/preprocessing-five-v1/datasets/S10/adaptive",
    "val": ROOT / "runs/preprocessing-five-v1/development/adaptive",
}

<a id="section-2"></a>

## 2. Offline Inputs

Both experiments retain identical IDs/GT/geometry. Each policy is applied to original pixels before encoding; nothing runs in the dataloader.

In [ ]:
if ALLOW_PREPARATION:
    prepared = prepare_data(ROOT, EXPERIMENT)
    print("Prepared:", prepared["path"], "Subset:", prepared["training_subset"])
else:
    require_paths(**{split: path / "data.yaml" for split, path in HISTORICAL.items()})
    _, rows = shared_splits(ROOT)
    for split, selected in populations(rows, "S10").items():
        directory = HISTORICAL[split]
        verify_alignment([directory], deep=False)
        export = read_json(directory / "export.json")
        expected_variant = "raw_reencoded" if config["variant"] == "raw" else "adaptive"
        assert export["variant"] == expected_variant
        assert export["recipe"] == ({} if config["variant"] == "raw" else config["recipes"])
        assert export["conversion"]["classes"] == ["car", "truck", "bus", "motorcycle", "bicycle"]
        assert export["encoding"]["jpeg_quality"] == 95
        metadata = pd.read_csv(directory / "metadata.csv")
        assert set(metadata.id) == {r["id"] for r in selected}
        display(metadata[["id", "width", "height"]].head())
    print("Read-only historical export inspection; nothing copied or relabeled as new.")

<a id="section-3"></a>

## 3. Preparation Summary

New exports are isolated from historical runs. An existing export with changed inputs fails; S20 is bound in data.yaml alongside the shared split checksum and seed.

In [ ]:
print("Experiment:", EXPERIMENT)
print("Historical measurements remain under their original paths.")
print("Training/evaluation are separate phases. Final holdout stays sealed.")